<a href="https://colab.research.google.com/github/ayeshayaz/python-practice/blob/main/Perlin_noise_flow_field.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from IPython.display import HTML, display

html_code = """
<canvas id="flow-canvas" width="700" height="520" style="background:#05060a;border-radius:12px;"></canvas>
<script>
(function(){
const canvas = document.getElementById('flow-canvas');
const ctx = canvas.getContext('2d');
const W = canvas.width, H = canvas.height;

function fade(t){return t*t*t*(t*(t*6-15)+10);}
function lerp(a,b,t){return a+t*(b-a);}
function grad(hash,x,y){
  const h = hash & 3;
  const u = h < 2 ? x : y;
  const v = h < 2 ? y : x;
  return ((h & 1) ? -u : u) + ((h & 2) ? -2*v : 2*v);
}
const perm = new Uint8Array(512);
(function(){
  const p = new Uint8Array(256);
  for(let i=0;i<256;i++) p[i]=i;
  for(let i=255;i>0;i--){
    const j = Math.floor(Math.random()*(i+1));
    const tmp = p[i]; p[i]=p[j]; p[j]=tmp;
  }
  for(let i=0;i<512;i++) perm[i]=p[i&255];
})();
function noise2(x,y){
  const X = Math.floor(x)&255, Y = Math.floor(y)&255;
  x -= Math.floor(x); y -= Math.floor(y);
  const u = fade(x), v = fade(y);
  const aa = perm[perm[X]+Y], ab = perm[perm[X]+Y+1];
  const ba = perm[perm[X+1]+Y], bb = perm[perm[X+1]+Y+1];
  return lerp(
    lerp(grad(aa,x,y), grad(ba,x-1,y), u),
    lerp(grad(ab,x,y-1), grad(bb,x-1,y-1), u),
    v
  );
}

const palette = ['#5DCAA5','#1D9E75','#0F6E56'];
const COUNT = 900;
const particles = [];
for(let i=0;i<COUNT;i++){
  particles.push({
    x: Math.random()*W, y: Math.random()*H,
    px: 0, py: 0, life: Math.random()*200,
    color: palette[Math.floor(Math.random()*palette.length)]
  });
}
particles.forEach(p=>{p.px=p.x;p.py=p.y;});

ctx.fillStyle = '#05060a';
ctx.fillRect(0,0,W,H);

let t = 0;
function step(){
  ctx.fillStyle = 'rgba(5,6,10,0.045)';
  ctx.fillRect(0,0,W,H);

  const scale = 0.0032;
  ctx.lineWidth = 1.1;
  ctx.lineCap = 'round';

  for(const p of particles){
    const n = noise2(p.x*scale, p.y*scale + t*0.08) * Math.PI * 4
            + noise2(p.x*scale*2.3, p.y*scale*2.3 - t*0.05) * Math.PI;
    const speed = 1.6;
    p.px = p.x; p.py = p.y;
    p.x += Math.cos(n) * speed;
    p.y += Math.sin(n) * speed;
    p.life -= 1;

    ctx.globalAlpha = 0.55;
    ctx.strokeStyle = p.color;
    ctx.beginPath();
    ctx.moveTo(p.px, p.py);
    ctx.lineTo(p.x, p.y);
    ctx.stroke();

    if(p.life <= 0 || p.x<0 || p.x>W || p.y<0 || p.y>H){
      p.x = Math.random()*W;
      p.y = Math.random()*H;
      p.px = p.x; p.py = p.y;
      p.life = 120 + Math.random()*200;
      p.color = palette[Math.floor(Math.random()*palette.length)];
    }
  }
  ctx.globalAlpha = 1;
  t += 0.016;
  requestAnimationFrame(step);
}
step();
})();
</script>
"""

display(HTML(html_code))